## Data Generation: IoU Scoring and Interpolation Sweeps

This notebook generates the core CSV artifacts used by downstream plotting and replacement experiments.

### Part 1
Compute IoU scores for 71 programs (70 golden programs + uniform lower diagonal) across BERT, GPT-2, and TinyLlama.

### Part 2
Run linear interpolation sweeps with k = 1..20 top-ranked programs per head.

### Output Files
- ../data/iou_scores_bert.csv
- ../data/iou_scores_gpt2.csv
- ../data/iou_scores_tinyllama.csv
- ../data/interpolation_k_bert.csv
- ../data/interpolation_k_gpt2.csv
- ../data/interpolation_k_tinyllama.csv

In [ ]:
# Install dependencies if needed
# !pip install torch transformers spacy nltk tqdm scikit-learn
# !python -m spacy download en_core_web_sm


In [1]:
import os
import csv
import json
import time
import traceback
import warnings
import inspect
import random
from pathlib import Path
from typing import Callable, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import spacy
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModel,
    PreTrainedModel, PreTrainedTokenizerBase
)
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Project-relative paths (notebook lives in code/)
DATA_DIR = "../data"
RESULTS_DIR = "../results"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

nlp = spacy.load("en_core_web_sm")
print(f"[INFO] Imports ready. DATA_DIR={DATA_DIR} | RESULTS_DIR={RESULTS_DIR}")

[INFO] Imports ready. DATA_DIR=../data | RESULTS_DIR=../results


In [2]:
# ── Scoring helpers ──────────────────────────────────────────────────────────

def iou_score(p: np.ndarray, q: np.ndarray) -> float:
    """IoU similarity between two attention distributions (higher = better)."""
    p = np.clip(p, 1e-12, 1.0)
    q = np.clip(q, 1e-12, 1.0)
    intersection = np.sum(np.minimum(p, q))
    union        = np.sum(np.maximum(p, q))
    return float(intersection / union) if union > 0 else 0.0


def score_head_iou(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    layer: int,
    head: int,
    pattern: Callable,
    sentence: str,
) -> float:
    """
    Returns mean row-wise IoU between actual attention and predicted pattern.
    Returns NaN on any error so callers can handle gracefully.
    """
    try:
        tokens = tokenizer(sentence, return_tensors="pt")
        with torch.no_grad():
            att = model(**tokens, output_attentions=True).attentions[layer][0, head].detach().numpy()

        # detect paradigm: 1-arg (Jacob's) vs 2-arg (sentence, tokenizer)
        n_params = len(inspect.signature(pattern).parameters)
        if n_params == 1:
            _, pred_att = pattern(sentence)
        else:
            _, pred_att = pattern(sentence, tokenizer)

        pred_att = np.array(pred_att, dtype=float)
        min_len = min(att.shape[0], pred_att.shape[0])
        att      = att[:min_len, :min_len]
        pred_att = pred_att[:min_len, :min_len]

        row_ious = [iou_score(att[i], pred_att[i]) for i in range(min_len)]
        return float(np.mean(row_ious))

    except Exception:
        return float("nan")


In [3]:
# --- Load golden programs + uniform_lower_diagonal ---
import sys
module_root = os.path.abspath("..")
if module_root not in sys.path:
    sys.path.append(module_root)

from data import golden_programs

all_programs: List[Callable] = [
    obj for _, obj in inspect.getmembers(golden_programs, inspect.isfunction)
]

# Save index mapping so CSV program_idx values are always interpretable
PROGRAM_INDEX_MAP = {i: p.__name__ for i, p in enumerate(all_programs)}
print(f"[INFO] Total programs loaded: {len(all_programs)}")

for idx, name in PROGRAM_INDEX_MAP.items():
    print(f"  {idx}: {name}")
print(f"[INFO] Saving program index map to {DATA_DIR}/program_index_map.json")

with open(f"{DATA_DIR}/program_index_map.json", "w", encoding="utf-8") as _f:
    json.dump(PROGRAM_INDEX_MAP, _f, indent=2)

print("[DONE] Program index map written.")

[INFO] Total programs loaded: 71
  0: adverbial_modulation
  1: appositive_phrase_attention
  2: cls_attention
  3: complement_adjunct_relationship
  4: compound_element_association
  5: compound_modifier_attention
  6: compound_word_attention_pattern
  7: conjunction_based_grouping
  8: conjunction_resolution
  9: contextual_anchoring
  10: coord_and_verb_dependency
  11: coordination_attention
  12: coreference_resolution
  13: dependencies
  14: dominant_subject_attention
  15: emphasize_verbs_and_objects
  16: eos_attention
  17: first_token_dominance
  18: first_token_domination
  19: first_token_emphasis
  20: head_initial_token_emphasis
  21: high_saliency_relationship_detection
  22: initial_contextual_attention
  23: initial_element_reinforcement
  24: initial_phrase_contextualization
  25: initial_phrase_dominance
  26: initial_reference_attention
  27: initial_token_anchoring
  28: initial_token_attachment
  29: initial_token_attention
  30: initial_token_centralization
  31

In [4]:
# --- Load generic sentences and sample 5 (seeded random sample) ---
df_json = pd.read_json(f"{DATA_DIR}/generic_sentences.json")
generic_sentences_all = df_json[0].tolist()

_rng = random.Random(RANDOM_SEED)
_shuffled = generic_sentences_all[:]
_rng.shuffle(_shuffled)
SAMPLE_SENTENCES: List[str] = _shuffled[:5]

# Save mapping so sentence_idx values in CSVs remain recoverable
SENTENCE_INDEX_MAP = {i: s for i, s in enumerate(SAMPLE_SENTENCES)}
print(f"[INFO] Total candidate sentences: {len(generic_sentences_all)}")
print(f"[INFO] Sample size: {len(SAMPLE_SENTENCES)}")
for idx, sentence in SENTENCE_INDEX_MAP.items():
    print(f"  {idx}: {sentence}")
print(f"[INFO] Saving sentence index map to {DATA_DIR}/sentence_index_map.json")

with open(f"{DATA_DIR}/sentence_index_map.json", "w", encoding="utf-8") as _f:
    json.dump(SENTENCE_INDEX_MAP, _f, indent=2)

print("[DONE] Sentence index map written.")

[INFO] Total candidate sentences: 100
[INFO] Sample size: 5
  0: The rhythmic crash of the waves, breaking against the shore, provided a soothing soundtrack to the quiet evening.
  1: He contemplated, for a moment, the vastness of the universe, its endless possibilities, and his tiny place within it.
  2: 'That's an excellent point!' she agreed, nodding enthusiastically, acknowledging the validity of his argument.
  3: If you truly want to succeed, you must work diligently, persevere through challenges, and never give up on your dreams.
  4: He meticulously organized his tools: wrenches, screwdrivers, pliers, and a tape measure, all in their designated spots.
[INFO] Saving sentence index map to ../data/sentence_index_map.json
[DONE] Sentence index map written.


In [5]:
# --- Model configurations ---
MODEL_CONFIGS = {
    "bert": {
        "model_name": "bert-base-uncased",
        "out_csv":    f"{DATA_DIR}/iou_scores_bert.csv",
    },
    "gpt2": {
        "model_name": "openai-community/gpt2",
        "out_csv":    f"{DATA_DIR}/iou_scores_gpt2.csv",
    },
    "tinyllama": {
        "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
        "out_csv":    f"{DATA_DIR}/iou_scores_tinyllama.csv",
    },
}

print("[INFO] Model configurations loaded.")

[INFO] Model configurations loaded.


In [12]:
from typing import Dict, List, Tuple, Callable
import numpy as np
from transformers import PreTrainedTokenizerBase

# ── Updated Category definitions ─────────────────────────────────────────────
# Restored missing programs and ensured mutual exclusivity (Total: 71)

PROGRAM_CATEGORIES: Dict[str, List[str]] = {
    "strict_anchors": [
        "initial_token_attention", "initial_token_anchoring", "cls_attention", 
        "first_token_domination", "first_token_emphasis", "head_initial_token_emphasis",
        "initial_word_attention", "sentence_start_attention", "sentence_beginning_salience",
        "initial_token_attachment", "initial_token_reference_attention", "initial_token_emphasis",
        "initial_token_centralization", "leading_contextual_emphasis", "sentence_start_dominance",
        "initial_phrase_contextualization", "initial_element_reinforcement", "token_reinforcement",
        "sentence_initiation_emphasis", "initial_token_dominance", "sentence_boundary_focus",
        "sentence_level_attention", "token_emphasis_subsequent_dominance", 
        "first_token_dominance", # Restored (was duplicate in OG, now unique)
        "compound_element_association", "conjunction_based_grouping", 
        "dominant_subject_attention", "initial_contextual_attention", 
        "initial_phrase_dominance", "negation_attention", "semantic_grouping",
        "sentence_beginning_attention_pattern", "sentence_beginning_emphasis",
        "sentence_level_initial_token_repetition", "sentence_opening_salience"
    ],
    "broadcasts_and_global": [
        "adverbial_modulation", "appositive_phrase_attention", "compound_modifier_attention",
        "compound_word_attention_pattern", "high_saliency_relationship_detection",
        "parenthetical_attention", "parenthetical_insertion", "quotation_association",
        "uniform_attention", "contextual_anchoring", "sentence_position_preference"
    ],
    "structural_and_boundaries": [
        "eos_attention", "last_token_attention", "special_token_attention",
        "sentence_initial_dominance", "punctuation_attention"
    ],
    "relative_flow": [
        "next_attention", "previous_attention", "relative_position_attention",
        "dependencies"
    ],
    "identity_and_local": [
        "same_attention", "repeated_attention", "lexical_diversity_focus", 
        "rare_word_dominance", "conjunction_resolution", "pos_alignment"
    ],
    "semantic_and_syntactic_spikes": [
        "initial_reference_attention", "main_subject_attention",
        "sentence_start_emphasis", "complement_adjunct_relationship", 
        "coord_and_verb_dependency", "coordination_attention", 
        "coreference_resolution", "emphasize_verbs_and_objects", "pronoun_reference"
    ],
    "global_uniform": [
        "uniform_lower_diagonal"
    ]
}

# ── Centroids ────────────────────────────────────────────────────────────────

def global_uniform_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """A flat, uniform distribution across all tokens."""
    n = len(tokenizer([sentence], return_tensors="pt").input_ids[0])
    out = np.ones((n, n)) / n
    return "Global Uniform Centroid", out

def cls_anchor_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """Pure CLS/Start token broadcast."""
    n = len(tokenizer([sentence], return_tensors="pt").input_ids[0])
    out = np.zeros((n, n))
    out[:, 0] = 1.0
    out /= np.clip(out.sum(axis=1, keepdims=True), a_min=1e-9, a_max=None)
    return "CLS Anchor Centroid", out

def eos_anchor_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """Pure EOS/End token broadcast."""
    n = len(tokenizer([sentence], return_tensors="pt").input_ids[0])
    out = np.zeros((n, n))
    out[:, -1] = 1.0
    out /= np.clip(out.sum(axis=1, keepdims=True), a_min=1e-9, a_max=None)
    return "EOS Anchor Centroid", out

def identity_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """Pure diagonal (self-attention) matrix."""
    n = len(tokenizer([sentence], return_tensors="pt").input_ids[0])
    out = np.eye(n)
    return "Identity Centroid", out

def dynamic_spike_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """Handles the Semantic/Syntactic Spike programs using a mixture of diagonal and subject-focus."""
    n = len(tokenizer([sentence], return_tensors="pt").input_ids[0])
    out = np.eye(n) * 0.4
    if n > 2:
        # Heavily weight the second and third tokens (typical subject/root positions)
        out[:, 1:min(3, n)] += 0.3
    out /= np.clip(out.sum(axis=1, keepdims=True), a_min=1e-9, a_max=None)
    return "Semantic/Syntactic Spike Centroid", out

def relative_flow_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """Targets off-diagonal 'flow' patterns (i+1, i-1)."""
    n = len(tokenizer([sentence], return_tensors="pt").input_ids[0])
    out = np.zeros((n, n))
    for i in range(n):
        if i > 0: out[i, i-1] = 0.6
        if i < n-1: out[i, i+1] = 0.4
    out /= np.clip(out.sum(axis=1, keepdims=True), a_min=1e-9, a_max=None)
    return "Relative Flow Centroid", out

def broadcasts_global_centroid(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]:
    """broadcasts_and_global: Uniform distribution across all tokens."""
    n = len(tokenizer([sentence], return_tensors="pt").input_ids[0])
    out = np.ones((n, n)) / n
    return "Broadcasts/Global Centroid", out

# ── Updated Registry ────────────────────────────────────────────────────────

CATEGORY_CENTROIDS: Dict[str, Callable] = {
    "strict_anchors":             cls_anchor_centroid,
    "broadcasts_and_global":      broadcasts_global_centroid,
    "structural_and_boundaries":  eos_anchor_centroid,
    "relative_flow":              relative_flow_centroid,
    "identity_and_local":         identity_centroid,
    "semantic_and_syntactic_spikes": dynamic_spike_centroid,
    "global_uniform":             global_uniform_centroid
}

# Reverse lookup: program name → category
PROGRAM_TO_CATEGORY: Dict[str, str] = {
    prog: cat for cat, progs in PROGRAM_CATEGORIES.items() for prog in progs
}

# ── Verification Logic ───────────────────────────────────────────────────────
# (Assumes all_programs list is defined elsewhere)
_name_to_idx = {fn.__name__: i for i, fn in enumerate(all_programs)}
CATEGORY_PROG_INDICES: Dict[str, List[int]] = {}

for cat, prog_names in PROGRAM_CATEGORIES.items():
    CATEGORY_PROG_INDICES[cat] = sorted([_name_to_idx[n] for n in prog_names if n in _name_to_idx])

total_mapped = sum(len(v) for v in CATEGORY_PROG_INDICES.values())
print(f"[INFO] 71-Program Coverage: {total_mapped}/71")

[INFO] 71-Program Coverage: 71/71


```markdown
after a couple iterations, i've attached a better PROGRAM_CATEGORIES you should check out (although there are a couple stragglers, strict_anchors, broadcasts_and_global, structural_and_boundaries, identity_and_local are all ~0.7 which is strong leaving less than 20 out of 71 programs without a good centroid). you can attempt to edit/add the categories for these remaining programs. next, i want you to change two things. i want to next run my experiment on llama3b which requires gpu and i'll use colab for that. so 1, i want a flag at the top to specify whether notebook is running locally or on colab which determines the data specifications (e.g., file paths, model names). 2, I want you to ensure that i can add llama3b to the list of models and the evaluation code will work on llama3b weights (one slight differences I think might include grouped query attention)automatically run the same experiments on it.
``` edit `write_data.ipynb` 

In [ ]:
[INFO] Within-category centroid coherence:
  Category                     N    mean     min     max
  strict_anchors              33   0.767   0.000   1.000
  broadcasts_and_global       11   0.804   0.385   1.000
  structural_and_boundaries    5   0.698   0.303   1.000
  relative_flow                4   0.202   0.042   0.391
  identity_and_local           6   0.764   0.242   0.957
  semantic_and_syntactic_sp    9   0.070   0.002   0.202
  global_uniform               1   0.409   0.409   0.409

In [13]:
# ── Category exploration: validate centroids and check placement ──────────────
# Run every program on the first sample sentence, compute IoU vs each centroid,
# and flag any program whose best centroid differs from its assigned category.
# Requires only a lightweight tokenizer (BERT) – no full model forward pass.

_probe_sentence = SAMPLE_SENTENCES[0]
_probe_tok      = AutoTokenizer.from_pretrained("bert-base-uncased")

# Pre-compute program matrices on probe sentence
print(f"[INFO] Probe: {_probe_sentence!r}")
print(f"[INFO] Computing {len(all_programs)} program outputs...")
prog_matrices_probe: Dict[int, np.ndarray] = {}
for p_idx, prog in enumerate(all_programs):
    try:
        n_params = len(inspect.signature(prog).parameters)
        _, mat   = prog(_probe_sentence) if n_params == 1 else prog(_probe_sentence, _probe_tok)
        prog_matrices_probe[p_idx] = np.array(mat, dtype=float)
    except Exception as e:
        print(f"  [WARN] {prog.__name__}: {e}")
print(f"[INFO] {len(prog_matrices_probe)} / {len(all_programs)} program matrices computed.")

# Pre-compute centroid matrices on probe sentence
centroid_matrices_probe: Dict[str, np.ndarray] = {}
for cat, fn in CATEGORY_CENTROIDS.items():
    _, cmat = fn(_probe_sentence, _probe_tok)
    centroid_matrices_probe[cat] = np.array(cmat, dtype=float)

categories_ordered = list(CATEGORY_CENTROIDS.keys())

# Per-program IoU vs every centroid
header = f"{'':1} {'Program':<42}" + "".join(f"  {c[:9]:>9}" for c in categories_ordered)
header += f"  {'Assigned':>18}  {'Best centroid':>22}  {'Match':>5}"
print("\n" + header)
print("-" * len(header))

mismatches = []
for p_idx, prog in enumerate(all_programs):
    if p_idx not in prog_matrices_probe:
        continue
    mat          = prog_matrices_probe[p_idx]
    assigned_cat = PROGRAM_TO_CATEGORY.get(prog.__name__, "UNCATEGORIZED")
    scores: Dict[str, float] = {}
    for cat, cmat in centroid_matrices_probe.items():
        min_len = min(mat.shape[0], cmat.shape[0])
        row_ious = [iou_score(mat[i, :min_len], cmat[i, :min_len]) for i in range(min_len)]
        scores[cat] = float(np.mean(row_ious))
    best_cat = max(scores, key=scores.__getitem__)
    match    = "✓" if best_cat == assigned_cat else "⚠"
    if best_cat != assigned_cat:
        mismatches.append((prog.__name__, assigned_cat, best_cat))
    row = f"{match} {prog.__name__:<42}"
    row += "".join(f"  {scores[c]:>9.3f}" for c in categories_ordered)
    row += f"  {assigned_cat:>18}  {best_cat:>22}  {match:>5}"
    print(row)

print(f"\n[INFO] Mismatches (best centroid ≠ assigned category): {len(mismatches)}")
for name, assigned, best in mismatches:
    print(f"  {name}: assigned={assigned}, best centroid={best}")

# Within-category centroid IoU summary
print("\n[INFO] Within-category centroid coherence:")
print(f"  {'Category':<25}  {'N':>3}  {'mean':>6}  {'min':>6}  {'max':>6}")
for cat, idxs in CATEGORY_PROG_INDICES.items():
    cmat   = centroid_matrices_probe[cat]
    ious   = []
    for p_idx in idxs:
        if p_idx not in prog_matrices_probe:
            continue
        mat = prog_matrices_probe[p_idx]
        min_len = min(mat.shape[0], cmat.shape[0])
        ious.append(float(np.mean([iou_score(mat[i, :min_len], cmat[i, :min_len]) for i in range(min_len)])))
    if ious:
        print(f"  {cat:<25}  {len(ious):>3}  {np.mean(ious):>6.3f}  {np.min(ious):>6.3f}  {np.max(ious):>6.3f}")
    else:
        print(f"  {cat:<25}  ---")


[INFO] Probe: 'The rhythmic crash of the waves, breaking against the shore, provided a soothing soundtrack to the quiet evening.'
[INFO] Computing 71 program outputs...
[INFO] 71 / 71 program matrices computed.

  Program                                     strict_an  broadcast  structura  relative_  identity_  semantic_  global_un            Assigned           Best centroid  Match
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
✓ adverbial_modulation                            0.107      0.915      0.020      0.042      0.064      0.072      0.915  broadcasts_and_global   broadcasts_and_global      ✓
✓ appositive_phrase_attention                     0.107      0.915      0.020      0.042      0.064      0.072      0.915  broadcasts_and_global   broadcasts_and_global      ✓
✓ cls_attention                                   1.000      0.022      0.000      0.019  

In [27]:
# ── PART 1: Category-aware IoU scoring ───────────────────────────────────────
#
# Speed-up strategy vs. the original 4-level nested loop:
#   Original:  p_idx × layer × head × sentence  → model(sentence) called once
#              per (p_idx, layer, head, sentence) = N_prog × N_layers × N_heads × N_sent
#              model forward passes.  For BERT that's 71 × 12 × 12 × 5 = 51 120 passes.
#
#   New:
#     Step 1 – cache all model attentions: N_sent forward passes total (5 for BERT).
#     Step 2 – cache all program matrices: N_prog × N_sent cheap numpy calls (355).
#     Step 3 – assign each (layer, head) to best centroid category.
#     Step 4 – score only programs in the assigned category (~avg 10 of 71).
#
#   Net: ~5 model passes + ~7× fewer scoring iterations ≈ order-of-magnitude faster.
#
# CSV schema unchanged: layer, head, program_idx, sentence_idx, iou_score
# Rows per head limited to programs in the matched category → Part 2 rankings
# are built only from within-category programs (by design).

from collections import Counter

PART1_COLUMNS = ["layer", "head", "program_idx", "sentence_idx", "iou_score"]


def load_done_keys(csv_path: str) -> set:
    """Return set of (layer, head, program_idx, sentence_idx) already present in CSV."""
    done = set()
    if not os.path.exists(csv_path):
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]),
                      int(row["program_idx"]), int(row["sentence_idx"])))
    except Exception as e:
        print(f"[WARN] Could not read existing CSV {csv_path}: {e}")
    return done


for model_key, cfg in MODEL_CONFIGS.items():
    model_name = cfg["model_name"]
    out_csv    = cfg["out_csv"]

    print(f"\n{'='*66}")
    print(f"[INFO] Model : {model_key}  ({model_name})")
    print(f"[INFO] Output: {out_csv}")
    print(f"{'='*66}")

    try:
        model_obj     = AutoModel.from_pretrained(model_name, output_attentions=True)
        tokenizer_obj = AutoTokenizer.from_pretrained(model_name)
        model_obj.eval()
    except Exception as e:
        print(f"[ERROR] Could not load model {model_name}: {e}")
        traceback.print_exc()
        continue

    num_layers = model_obj.config.num_hidden_layers
    num_heads  = model_obj.config.num_attention_heads
    print(f"[INFO] Layers={num_layers} | Heads={num_heads} | Programs={len(all_programs)}")

    # ── Step 1: cache all attention tensors (one forward pass per sentence) ────
    print("[INFO] Pre-computing model attentions …")
    # attn_cache[s_idx][layer][head] → np.ndarray(seq_len, seq_len) | None sentinel
    attn_cache: List = []
    for s_idx, sentence in enumerate(SAMPLE_SENTENCES):
        try:
            tokens = tokenizer_obj(sentence, return_tensors="pt")
            with torch.no_grad():
                outputs = model_obj(**tokens, output_attentions=True)
            layer_heads = [
                [outputs.attentions[l][0, h].detach().numpy() for h in range(num_heads)]
                for l in range(num_layers)
            ]
            attn_cache.append(layer_heads)
        except Exception as e:
            print(f"  [WARN] s_idx={s_idx} failed: {e}")
            attn_cache.append(None)
    n_ok = sum(1 for x in attn_cache if x is not None)
    print(f"[INFO] Attention cache ready ({n_ok}/{len(SAMPLE_SENTENCES)} sentences OK).")

    # ── Step 2: cache all program outputs (pure numpy, no model) ──────────────
    print("[INFO] Pre-computing program outputs …")
    # prog_cache[p_idx][s_idx] → np.ndarray | None
    prog_cache: List[List] = []
    for p_idx, prog in enumerate(all_programs):
        prog_outputs = []
        for s_idx, sentence in enumerate(SAMPLE_SENTENCES):
            try:
                n_params = len(inspect.signature(prog).parameters)
                _, mat   = prog(sentence) if n_params == 1 else prog(sentence, tokenizer_obj)
                prog_outputs.append(np.array(mat, dtype=float))
            except Exception:
                prog_outputs.append(None)
        prog_cache.append(prog_outputs)
    print("[INFO] Program cache ready.")

    # ── Step 3: cache centroid matrices ──────────────────────────────────────
    print("[INFO] Pre-computing centroid matrices …")
    # centroid_cache[cat][s_idx] → np.ndarray | None
    centroid_cache: Dict[str, List] = {cat: [] for cat in CATEGORY_CENTROIDS}
    for cat, centroid_fn in CATEGORY_CENTROIDS.items():
        for s_idx, sentence in enumerate(SAMPLE_SENTENCES):
            try:
                _, cmat = centroid_fn(sentence, tokenizer_obj)
                centroid_cache[cat].append(np.array(cmat, dtype=float))
            except Exception:
                centroid_cache[cat].append(None)

    # ── Step 4: assign each (layer, head) to best category via centroid IoU ───
    print("[INFO] Assigning heads to categories …")
    head_category: Dict[Tuple[int, int], str] = {}
    cats = list(CATEGORY_CENTROIDS.keys())

    for layer in range(num_layers):
        for head in range(num_heads):
            cat_scores: Dict[str, List[float]] = {c: [] for c in cats}
            for s_idx in range(len(SAMPLE_SENTENCES)):
                if attn_cache[s_idx] is None:
                    continue
                att = attn_cache[s_idx][layer][head]
                for cat in cats:
                    cmat = centroid_cache[cat][s_idx]
                    if cmat is None:
                        continue
                    min_len = min(att.shape[0], cmat.shape[0])
                    row_ious = [iou_score(att[i, :min_len], cmat[i, :min_len])
                                for i in range(min_len)]
                    cat_scores[cat].append(float(np.mean(row_ious)))
            mean_scores = {c: float(np.mean(v)) if v else 0.0 for c, v in cat_scores.items()}
            head_category[(layer, head)] = max(mean_scores, key=mean_scores.__getitem__)

    cat_dist = Counter(head_category.values())
    total_full = num_layers * num_heads * len(all_programs) * len(SAMPLE_SENTENCES)
    total_cat  = sum(
        len(CATEGORY_PROG_INDICES.get(head_category[(l, h)], [])) * len(SAMPLE_SENTENCES)
        for l in range(num_layers) for h in range(num_heads)
    )
    print("[INFO] Head-to-category assignment:")
    for cat in cats:
        n_progs = len(CATEGORY_PROG_INDICES.get(cat, []))
        print(f"  {cat:<25}: {cat_dist.get(cat, 0):>3} heads  |  {n_progs} programs")
    print(f"[INFO] Rows to score: {total_cat:,}  (vs {total_full:,} full → "
          f"{100 * total_cat / total_full:.1f}% of original)")

    # ── Step 5: score within-category programs and stream to CSV ─────────────
    done_keys  = load_done_keys(out_csv)
    write_header = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
    errors     = 0
    done_count = len(done_keys)
    t0         = time.time()

    with open(out_csv, "a", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        if write_header:
            writer.writerow(PART1_COLUMNS)
            fh.flush()

        for layer in range(num_layers):
            for head in range(num_heads):
                assigned_cat = head_category[(layer, head)]
                p_indices    = CATEGORY_PROG_INDICES.get(assigned_cat, list(range(len(all_programs))))

                for p_idx in p_indices:
                    for s_idx in range(len(SAMPLE_SENTENCES)):
                        key = (layer, head, p_idx, s_idx)
                        if key in done_keys:
                            done_count += 1
                            continue

                        att  = attn_cache[s_idx][layer][head] if attn_cache[s_idx] else None
                        pred = prog_cache[p_idx][s_idx]

                        if att is None or pred is None:
                            score = float("nan")
                        else:
                            min_len  = min(att.shape[0], pred.shape[0])
                            row_ious = [iou_score(att[i, :min_len], pred[i, :min_len])
                                        for i in range(min_len)]
                            score    = float(np.mean(row_ious))

                        score_out = round(score, 3) if not np.isnan(score) else score
                        writer.writerow([layer, head, p_idx, s_idx, score_out])
                        fh.flush()
                        done_count += 1

                        if np.isnan(score):
                            errors += 1

                        if done_count % 500 == 0:
                            elapsed = time.time() - t0
                            pct     = done_count / total_cat * 100 if total_cat else 0
                            eta     = elapsed / done_count * (total_cat - done_count) if done_count else 0
                            print(
                                f"[INFO] [{model_key}] {done_count}/{total_cat} ({pct:.1f}%) | "
                                f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | NaN={errors}"
                            )

    del model_obj, tokenizer_obj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"[DONE] {model_key} complete. NaN scores: {errors}")
    df_check = pd.read_csv(out_csv)
    print(f"[INFO] Rows in CSV: {len(df_check)}")

print("\n[DONE] Part 1 complete.")



[INFO] Model : bert  (bert-base-uncased)
[INFO] Output: ../data/iou_scores_bert.csv
[INFO] Layers=12 | Heads=12 | Programs=71
[INFO] Pre-computing model attentions …
[INFO] Attention cache ready (5/5 sentences OK).
[INFO] Pre-computing program outputs …
[INFO] Program cache ready.
[INFO] Pre-computing centroid matrices …
[INFO] Assigning heads to categories …


KeyboardInterrupt: 

In [10]:
# --- Part 1 verification ---
for model_key, cfg in MODEL_CONFIGS.items():
    csv_path = cfg["out_csv"]
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(
            f"[INFO] {model_key}: rows={len(df)} | NaN iou={df['iou_score'].isna().sum()} | "
            f"programs={df['program_idx'].nunique()} | mean_iou={df['iou_score'].mean():.4f}"
        )
    else:
        print(f"[WARN] {model_key}: CSV not found at {csv_path}")

[INFO] bert: rows=51120 | NaN iou=0 | programs=71 | mean_iou=0.1090
[INFO] gpt2: rows=51120 | NaN iou=288 | programs=71 | mean_iou=0.2613
[INFO] tinyllama: rows=237401 | NaN iou=0 | programs=68 | mean_iou=0.2909


## Part 2: Linear Interpolation Sweep (k = 1..20)

For each model and each attention head:
1. Rank programs by Part 1 mean IoU
2. Fit linear combinations using the top-k programs
3. Record IoU for each k

Output columns:
- model_key
- layer
- head
- k
- iou_score
- program_idxs_used

In [ ]:
# --- Part 2 helpers ---

def linear_interp_iou(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    layer: int,
    head: int,
    top_k_programs: List[Callable],
    sentences: List[str],
) -> float:
    """
    Fit a linear combination of top_k_programs to minimize squared error vs
    actual attention, then measure mean IoU of the fitted combination.
    Returns NaN on failure.
    """
    all_ious = []

    for sentence in sentences:
        try:
            tokens = tokenizer(sentence, return_tensors="pt")
            with torch.no_grad():
                att = model(**tokens, output_attentions=True).attentions[layer][0, head].detach().numpy()

            # Build feature matrix: each program contributes one flattened attention matrix
            X_rows = []
            for prog in top_k_programs:
                try:
                    n_params = len(inspect.signature(prog).parameters)
                    _, pred = prog(sentence) if n_params == 1 else prog(sentence, tokenizer)
                    pred = np.array(pred, dtype=float)
                    min_len = min(att.shape[0], pred.shape[0])
                    pred = pred[:min_len, :min_len]
                    X_rows.append(pred.flatten())
                except Exception:
                    return float("nan")

            min_len = min(att.shape[0], len(X_rows[0]) ** 0.5)
            seq_len = att.shape[0]

            flat_len = seq_len * seq_len
            X_mat = np.array([r[:flat_len] for r in X_rows]).T
            y_vec = att.flatten()[:flat_len]

            X_mat = np.nan_to_num(X_mat)
            y_vec = np.nan_to_num(y_vec)

            reg = LinearRegression(fit_intercept=True).fit(X_mat, y_vec)
            fitted = reg.intercept_ + X_mat @ reg.coef_
            fitted = fitted.reshape(seq_len, seq_len)

            row_sums = fitted.sum(axis=1, keepdims=True)
            row_sums = np.where(row_sums == 0, 1, row_sums)
            fitted = fitted / row_sums
            fitted = np.clip(fitted, 1e-12, None)

            row_ious = [iou_score(att[i], fitted[i]) for i in range(seq_len)]
            all_ious.append(float(np.mean(row_ious)))

        except Exception:
            all_ious.append(float("nan"))

    valid = [v for v in all_ious if not np.isnan(v)]
    return float(np.mean(valid)) if valid else float("nan")


K_VALUES = list(range(1, 21))
PART2_COLUMNS = ["model_key", "layer", "head", "k", "iou_score", "program_idxs_used"]

print(f"[INFO] Part 2 K values: {K_VALUES}")
print(f"[INFO] Part 2 columns: {PART2_COLUMNS}")

In [ ]:
# --- PART 2: Linear interpolation sweep ---
# Strategy:
# 1) Use Part 1 CSV to rank programs per head by mean IoU
# 2) For each head and k in 1..20, fit linear combinations of top-k programs
# 3) Stream each row to CSV
# Resume-safe: skips (layer, head, k) that are already present.

INTERP_CONFIGS = {
    "bert":      f"{DATA_DIR}/interpolation_k_bert.csv",
    "gpt2":      f"{DATA_DIR}/interpolation_k_gpt2.csv",
    "tinyllama": f"{DATA_DIR}/interpolation_k_tinyllama.csv",
}


def load_done_interp(csv_path: str) -> set:
    done = set()
    if not os.path.exists(csv_path):
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]), int(row["k"])))
    except Exception as e:
        print(f"[WARN] Could not read {csv_path}: {e}")
    return done


for model_key in MODEL_CONFIGS:
    iou_csv = MODEL_CONFIGS[model_key]["out_csv"]
    out_csv = INTERP_CONFIGS[model_key]
    model_name = MODEL_CONFIGS[model_key]["model_name"]

    print(f"\n[INFO] {'=' * 64}")
    print(f"[INFO] Model: {model_key} ({model_name})")
    print(f"[INFO] Input Part 1 CSV: {iou_csv}")
    print(f"[INFO] Output Part 2 CSV: {out_csv}")
    print(f"[INFO] {'=' * 64}")

    if not os.path.exists(iou_csv):
        print(f"[WARN] Missing Part 1 CSV: {iou_csv}. Skipping.")
        continue

    iou_df = pd.read_csv(iou_csv)
    if iou_df.empty:
        print(f"[WARN] Part 1 CSV is empty for {model_key}. Skipping.")
        continue

    try:
        model_obj = AutoModel.from_pretrained(model_name, output_attentions=True)
        tokenizer_obj = AutoTokenizer.from_pretrained(model_name)
        model_obj.eval()
    except Exception as e:
        print(f"[ERROR] Could not load model {model_name}: {e}")
        traceback.print_exc()
        continue

    num_layers = model_obj.config.num_hidden_layers
    num_heads = model_obj.config.num_attention_heads

    # Mean IoU per (layer, head, program_idx), then rank programs per head
    head_rankings = {}
    grouped = (
        iou_df.groupby(["layer", "head", "program_idx"])["iou_score"]
        .mean()
        .reset_index()
    )
    for (layer, head), grp in grouped.groupby(["layer", "head"]):
        sorted_grp = grp.sort_values("iou_score", ascending=False)
        ordered_fns = []
        for p_idx in sorted_grp["program_idx"]:
            fn = all_programs[int(p_idx)] if int(p_idx) < len(all_programs) else None
            if fn is not None:
                ordered_fns.append(fn)
        head_rankings[(int(layer), int(head))] = ordered_fns

    total = num_layers * num_heads * len(K_VALUES)
    done_keys = load_done_interp(out_csv)
    print(f"[INFO] Resume keys loaded: {len(done_keys)} | Total target rows: {total}")

    write_header = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
    errors = 0
    done_count = len(done_keys)
    t0 = time.time()

    with open(out_csv, "a", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        if write_header:
            writer.writerow(PART2_COLUMNS)
            fh.flush()

        for layer in range(num_layers):
            for head in range(num_heads):
                ranked_fns = head_rankings.get((layer, head), all_programs)

                for k in K_VALUES:
                    key = (layer, head, k)
                    if key in done_keys:
                        done_count += 1
                        continue

                    top_k_fns = ranked_fns[:k]
                    if not top_k_fns:
                        top_k_fns = all_programs[:k]

                    iou = linear_interp_iou(
                        model_obj,
                        tokenizer_obj,
                        layer,
                        head,
                        top_k_fns,
                        SAMPLE_SENTENCES,
                    )

                    prog_idx_used = "|".join(str(all_programs.index(f)) for f in top_k_fns)
                    iou_out = round(iou, 6) if not np.isnan(iou) else iou
                    writer.writerow([model_key, layer, head, k, iou_out, prog_idx_used])
                    fh.flush()
                    done_count += 1

                    if np.isnan(iou):
                        errors += 1

                    if done_count % 200 == 0:
                        elapsed = time.time() - t0
                        pct = done_count / total * 100
                        eta = elapsed / done_count * (total - done_count) if done_count else 0
                        print(
                            f"[INFO] [{model_key}] {done_count}/{total} ({pct:.1f}%) | "
                            f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | NaN={errors}"
                        )

    del model_obj, tokenizer_obj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"[DONE] {model_key} interpolation complete. NaN scores: {errors}")
    df_check = pd.read_csv(out_csv)
    print(f"[INFO] Rows currently in CSV: {len(df_check)}")

print("\n[DONE] Part 2 complete.")

In [13]:
# --- Final verification summary ---
print("[INFO] === Part 1: IoU Scores ===")
for model_key, cfg in MODEL_CONFIGS.items():
    p = cfg["out_csv"]
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(
            f"[INFO] {model_key}: rows={len(df)} | programs={df['program_idx'].nunique()} | "
            f"mean_iou={df['iou_score'].mean():.4f} | NaN={df['iou_score'].isna().sum()}"
        )
    else:
        print(f"[WARN] {model_key}: file not found at {p}")

print("\n[INFO] === Part 2: Interpolation Sweep ===")
for model_key, p in INTERP_CONFIGS.items():
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(
            f"[INFO] {model_key}: rows={len(df)} | k_range={df['k'].min()}..{df['k'].max()} | "
            f"mean_iou={df['iou_score'].mean():.4f} | NaN={df['iou_score'].isna().sum()}"
        )
    else:
        print(f"[WARN] {model_key}: file not found at {p}")

print("\n[DONE] Verification summary complete.")

[INFO] === Part 1: IoU Scores ===
[INFO] bert: rows=51120 | programs=71 | mean_iou=0.1090 | NaN=0
[INFO] gpt2: rows=51120 | programs=71 | mean_iou=0.2613 | NaN=288
[INFO] tinyllama: rows=237401 | programs=68 | mean_iou=0.2909 | NaN=0

[INFO] === Part 2: Interpolation Sweep ===
[INFO] bert: rows=2880 | k_range=1..20 | mean_iou=0.4760 | NaN=0
[INFO] gpt2: rows=2880 | k_range=1..20 | mean_iou=0.5893 | NaN=8
[INFO] tinyllama: rows=13088 | k_range=1..20 | mean_iou=0.6961 | NaN=0

[DONE] Verification summary complete.
